In [ ]:


!pip -q install gradio pypdf sentence-transformers scikit-learn requests


In [ ]:

import os
import time
import subprocess
import requests

# Install Ollama
install = subprocess.run(
    "curl -fsSL https://ollama.com/install.sh | sh",
    shell=True,
    capture_output=True,
    text=True
)

if install.returncode != 0:
    print("Installation error:")
    print(install.stderr)
else:
    print("Ollama installed successfully!")

# Check installation
result = subprocess.run(
    ["which", "ollama"],
    capture_output=True,
    text=True
)

print("Ollama path:", result.stdout.strip())

# Start server only if installation succeeded
if result.returncode == 0:
    ollama_process = subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    OLLAMA_URL = "http://127.0.0.1:11434"

    for i in range(30):
        try:
            response = requests.get(OLLAMA_URL, timeout=2)
            if response.status_code == 200:
                print("Ollama server is running!")
                break
        except requests.RequestException:
            pass

        time.sleep(2)
    else:
        print("Server did not start. Please share the error.")
else:
    print("Ollama installation failed. Please share the output above.")


Installation error:
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local

Ollama path: 
Ollama installation failed. Please share the output above.


In [6]:

import subprocess
import requests

print("Checking Ollama...")

try:
    r = requests.get("http://127.0.0.1:11434", timeout=5)
    print("Ollama status:", r.status_code)
except Exception as e:
    print("Ollama is not responding:", e)

result = subprocess.run(
    ["which", "ollama"],
    capture_output=True,
    text=True
)
print("Ollama path:", result.stdout.strip())

Checking Ollama...
Ollama is not responding: HTTPConnectionPool(host='127.0.0.1', port=11434): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7d22892dc910>: Failed to establish a new connection: [Errno 111] Connection refused'))
Ollama path: 


In [7]:

import gradio as gr
import numpy as np
import os
import requests

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Loading embedding model...")

embedder = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")
print("Ready for the next step.")


Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!
Ready for the next step.


In [8]:

# Sample document for testing the RAG chatbot

sample_notes = """
RAG stands for Retrieval-Augmented Generation.

RAG combines information retrieval with language generation.
It retrieves relevant information from documents before
sending the information to a language model.

The main steps of RAG are:
1. Document loading
2. Text splitting
3. Embedding generation
4. Similarity search
5. Answer generation

Embeddings represent text as numerical vectors.
Similarity search finds text related to a user's question.

Ollama allows users to run supported AI models locally.
RAG chatbots can answer questions from study notes,
PDF documents, manuals, and knowledge bases.

Python is a popular programming language used in AI,
machine learning, automation, and web development.
"""

# Split the text into chunks
def split_text(text, chunk_size=100, overlap=20):
    words = text.split()
    chunks = []

    for start in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[start:start + chunk_size])
        if chunk:
            chunks.append(chunk)

    return chunks

# Create document chunks
document_chunks = split_text(sample_notes)

# Convert chunks into embeddings
document_embeddings = embedder.encode(
    document_chunks,
    convert_to_numpy=True
)

document_name = "Sample study notes"

print("Sample notes loaded successfully!")
print("Number of chunks:", len(document_chunks))
print("Embeddings shape:", document_embeddings.shape)
print("\nFirst chunk:")
print(document_chunks[0])


Sample notes loaded successfully!
Number of chunks: 2
Embeddings shape: (2, 384)

First chunk:
RAG stands for Retrieval-Augmented Generation. RAG combines information retrieval with language generation. It retrieves relevant information from documents before sending the information to a language model. The main steps of RAG are: 1. Document loading 2. Text splitting 3. Embedding generation 4. Similarity search 5. Answer generation Embeddings represent text as numerical vectors. Similarity search finds text related to a user's question. Ollama allows users to run supported AI models locally. RAG chatbots can answer questions from study notes, PDF documents, manuals, and knowledge bases. Python is a popular programming language used in AI, machine learning, automation, and web development.


In [9]:

# CELL 6: Upload and index a PDF or TXT document

def upload_document(file):
    global document_chunks
    global document_embeddings
    global document_name

    if file is None:
        return "Please upload a PDF or TXT file."

    try:
        # Get uploaded file path
        path = file if isinstance(file, str) else file.name

        # Read PDF or TXT
        if path.lower().endswith(".pdf"):
            reader = PdfReader(path)

            text = "\n".join(
                page.extract_text() or ""
                for page in reader.pages
            )

        elif path.lower().endswith(".txt"):
            with open(path, "r", encoding="utf-8") as f:
                text = f.read()

        else:
            return "Please upload a PDF or TXT file."

        if not text.strip():
            return "No readable text found in this document."

        # Split document into chunks
        document_chunks = split_text(text)

        # Generate embeddings
        document_embeddings = embedder.encode(
            document_chunks,
            convert_to_numpy=True
        )

        # Store document name
        import os
        document_name = os.path.basename(path)

        return (
            "Document uploaded and indexed successfully!\n"
            f"File: {document_name}\n"
            f"Total chunks: {len(document_chunks)}"
        )

    except Exception as e:
        return f"Error processing document: {str(e)}"


print("Document upload function is ready!")


Document upload function is ready!


In [10]:

# CELL 7: RAG question-answering function

def ask_rag(question, history=None):

    if not question or not question.strip():
        return "Please enter a question."

    if len(document_chunks) == 0:
        return "Please upload a document first."

    try:
        # Step 1: Convert the question into an embedding
        query_embedding = embedder.encode(
            [question],
            convert_to_numpy=True
        )

        # Step 2: Find similarities with document chunks
        similarities = cosine_similarity(
            query_embedding,
            document_embeddings
        )[0]

        # Step 3: Retrieve the 3 most relevant chunks
        top_indices = np.argsort(similarities)[-3:][::-1]

        context = "\n\n".join(
            document_chunks[i] for i in top_indices
        )

        # Step 4: Prepare prompt for Ollama
        prompt = f"""
You are a helpful document question-answering assistant.

Answer the question using the context below.
If the answer is not present in the context,
say that you could not find it in the document.
Do not invent facts.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

        # Step 5: Generate answer using Ollama
        response = requests.post(
            f"{OLLAMA_URL}/api/generate",
            json={
                "model": MODEL,
                "prompt": prompt,
                "stream": False,
                "options": {"temperature": 0.2}
            },
            timeout=240
        )

        response.raise_for_status()

        answer = response.json()["response"]

        return (
            answer
            + f"\n\nSource document: {document_name}"
        )

    except Exception as e:
        return f"Error generating answer: {str(e)}"


print("RAG question-answering function is ready!")


RAG question-answering function is ready!


In [12]:

import gradio as gr

with gr.Blocks() as demo:
    gr.Markdown("# Mini RAG + Ollama Chatbot")

    file_input = gr.File(
        label="Upload PDF or TXT",
        file_types=[".pdf", ".txt"],
        type="filepath"
    )

    upload_button = gr.Button("Upload and Index")
    status = gr.Textbox(label="Status")

    upload_button.click(
        fn=upload_document,
        inputs=file_input,
        outputs=status
    )

    question = gr.Textbox(
        label="Ask a question",
        placeholder="What is RAG?"
    )

    ask_button = gr.Button("Get Answer")
    answer = gr.Textbox(label="AI Answer", lines=6)

    ask_button.click(
        fn=ask_rag,
        inputs=question,
        outputs=answer
    )

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://aa1dcd6bd9be4da2c3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
